# 04 — Sparse autoencoders: the residual stream in a readable basis

Notebooks 00–03 answered **"what circuit computes IOI?"** — a specific set of
heads, found by patching and ablation. This notebook asks a different question:
**"what does the residual stream *encode*, in a human-readable basis?"**

The obstacle is **superposition**: GPT-2's residual stream is 768-dimensional but
represents far more than 768 features, so it packs them in non-orthogonally.
Individual directions (and neurons) come out **polysemantic** — one direction
fires for several unrelated concepts. You can't just read off "the Mary
direction."

A **sparse autoencoder (SAE)** is the standard tool for pulling them apart. It
learns an *overcomplete* dictionary — here `d_model 768 → d_sae ≈ 24576` (32×) —
with an L1 penalty that forces only a handful of features active per token. The
hope (and largely the result): each feature is **monosemantic** — one
human-nameable thing.

We won't *train* one — a good SAE is a GPU-day and a project of its own. We load a
pretrained SAE (Joseph Bloom's `gpt2-small-res-jb`, one of the most-studied
releases, every feature browsable on [Neuronpedia](https://neuronpedia.org)), run
it on the IOI prompt, see what lights up, and tie it back to the circuit.

> **A note on checkpoints.** Unlike 00–03, the *specific* feature indices you get
> depend on the SAE release and your exact run. The code below is exact; treat the
> numeric expectations as ballparks and **interpret whatever your run surfaces** —
> that exploratory step *is* the skill here.

## Setup

`sae-lens` provides the pretrained SAEs and the `encode`/`decode` API. APIs in
this library move fast — the version is pinned in `requirements.txt`; if
`from_pretrained`'s return shape differs, the defensive unpack below handles it.

We load the SAE for **`blocks.9.hook_resid_pre`** — layer 9, exactly where the
name movers (9.6, 9.9) read their input. If the circuit's signal is anywhere in a
readable basis, it should be here.

In [ ]:
import sys
sys.path.insert(0, "..")

import torch
from interp_lab import load_model, ioi_task, logit_diff
from sae_lens import SAE

model = load_model()
task = ioi_task(model)
device = model.cfg.device

# from_pretrained has returned both `sae` and `(sae, cfg, sparsity)` across
# versions — unpack defensively.
loaded = SAE.from_pretrained("gpt2-small-res-jb", "blocks.9.hook_resid_pre", device=device)
sae = loaded[0] if isinstance(loaded, tuple) else loaded
HOOK = sae.cfg.hook_name
print(f"SAE hook: {HOOK}   d_in={sae.cfg.d_in}   d_sae={sae.cfg.d_sae}  "
      f"({sae.cfg.d_sae // sae.cfg.d_in}× overcomplete)")

## Exercise 1 — is the SAE faithful?

Before reading anything *off* the SAE, check it actually reconstructs the
activation it claims to decompose. Encode the layer-9 residual stream into
features, decode it back, and measure how much variance survives. A faithful SAE
reconstructs the resid closely (and, spliced into the model, recovers most of the
original loss).

**Predict.** Variance explained near 1.0, or noticeably lossy? (An SAE trades a
little reconstruction for a lot of sparsity — what's the going rate?)

In [ ]:
_, cache = model.run_with_cache(task.clean_tokens, names_filter=HOOK)
resid = cache[HOOK]                      # [batch, pos, d_model]

# TODO: feats = sae.encode(resid); recon = sae.decode(feats)
#       variance_explained = 1 - Var(resid - recon) / Var(resid)
...

<details><summary><b>Solution — Exercise 1</b></summary>

```python
feats = sae.encode(resid)                # [batch, pos, d_sae]
recon = sae.decode(feats)                # [batch, pos, d_model]
resid_err = resid - recon
var_explained = 1 - resid_err.var() / resid.var()
print(f"variance explained: {var_explained.item():.3f}")
print(f"mean active features / token: {(feats[0] > 0).float().sum(-1).mean():.1f}")
```

**Checkpoint.** Variance explained lands high — typically **~0.8–0.95** for these
residual SAEs — while only a few **dozen** of the 24576 features are active per
token. That gap (thousands of dimensions, dozens active) is the sparsity the L1
bought you. If reconstruction were poor, nothing below would be trustworthy; it
isn't, so read on.
</details>

## Exercise 2 — what fires at the IO position?

Now the payoff. Encode the IOI prompt and look at which features are active at the
**IO position** (`" Mary"`, the answer the name movers copy) and at **END**. With
~24576 features and only dozens active, the top handful at each position is a short,
inspectable list.

**Predict.** At the IO position, do you expect a "name / proper-noun" flavour of
feature? At END (where the prediction is formed), something more like
"about-to-emit-a-name"?

In [ ]:
feats = sae.encode(resid)[0]             # [pos, d_sae], batch 0

# TODO: for IO_POS and END, print the top-k feature indices + activations
#       (feats[pos].topk(k))
K = 8
...

<details><summary><b>Solution — Exercise 2</b></summary>

```python
for name, pos in [("IO (Mary)", task.IO_POS), ("END (to)", task.END)]:
    top = feats[pos].topk(K)
    print(f"\n{name}  pos={pos}")
    for val, idx in zip(top.values, top.indices):
        print(f"  feature {idx.item():>6}   act {val.item():.2f}")
```

**Checkpoint.** A short list per position — sparsity made visible. The *indices*
are what you carry into the next exercise; don't expect them to match anyone
else's run-for-run, but the **shape** (a few strong features, a long zero tail) is
universal. Interpreting those indices is Exercise 3.
</details>

## Exercise 3 — name the features (Neuronpedia)

A feature index is meaningless until you see what it fires on. Every feature in
this release has a [Neuronpedia](https://neuronpedia.org) dashboard — top
activating examples, an auto-interpretation, logit effects. For layer `L` the URL
is `neuronpedia.org/gpt2-small/{L}-res-jb/{feature_idx}`.

Build the links for your top IO-position features and actually open a few.

**Predict.** Among the top features at `" Mary"`, what do you expect to find —
a generic "first-name token" feature, a "the name Mary specifically" feature, a
"duplicated/earlier-seen token" feature, or position information?

In [ ]:
LAYER = 9
def neuronpedia_url(feature_idx, layer=LAYER):
    return f"https://neuronpedia.org/gpt2-small/{layer}-res-jb/{feature_idx}"

# TODO: print neuronpedia_url(...) for the top few features at task.IO_POS
...

<details><summary><b>Solution — Exercise 3</b></summary>

```python
for idx in feats[task.IO_POS].topk(5).indices:
    print(neuronpedia_url(idx.item()))
```

**Reading.** Open them. You'll commonly find a mix: **proper-noun / first-name**
features, features for **specific frequent names**, and — the interesting ones for
us — features that fire on a **token seen earlier in the context** (the SAE's view
of the same "duplicate" signal notebook 03 found mechanistically). The map between
"SAE feature" and "circuit component" is rarely 1:1, but where it lines up, you're
seeing the same computation from two directions: heads (how) and features (what).
</details>

## Exercise 4 — steer a feature, move the logit diff *(stretch)*

The honest causal test, mirroring the patching from phase 1: if a feature really
*encodes* something the circuit uses, **adding its decoder direction** into the
residual stream should move the behavior. Pick a name-ish feature from Exercise 3,
add `coeff · sae.W_dec[feature]` at the SAE hook point, and read the logit diff.

**Predict.** Amplify a "Mary / indirect-object" feature: logit diff up or down?
Amplify a "John / subject" feature?

In [ ]:
def steer(feature_idx, coeff):
    direction = sae.W_dec[feature_idx]          # [d_model], unit-ish decoder vector
    def hook(act, hook):                        # act: [batch, pos, d_model]
        act[:, :, :] = act + coeff * direction
        return act
    return logit_diff(model.run_with_hooks(
        task.clean_tokens, fwd_hooks=[(HOOK, hook)]), task)

# TODO: pick a feature idx from Ex 3; sweep coeff in e.g. [-20, -10, 0, 10, 20]
...

<details><summary><b>Solution — Exercise 4</b></summary>

```python
FEATURE = feats[task.IO_POS].topk(1).indices.item()   # or one you identified by name
for coeff in [-20, -10, 0, 10, 20]:
    print(f"coeff {coeff:+3d}   logit_diff {steer(FEATURE, coeff):+.3f}")
```

**Reading.** Steering a genuinely IO/name-related feature shifts the logit diff
monotonically with the coefficient — direct evidence the feature *participates* in
the behavior, not just correlates with it. Effect sizes vary by feature and layer;
a feature that does nothing is itself informative (it's not on the IOI path).
Caveat: adding the *same* direction at every position is blunt — for a cleaner
test, gate the hook to a single position, the SAE-basis analogue of the per-
position patching in notebook 01.
</details>

## Where this leaves the lab

Two lenses on one model, now both in hand:

- **Circuits** (00–03): *which components* compute a behavior — name movers,
  S-inhibition, duplicate/induction heads — found by intervention.
- **Features** (04): *what the activations encode* in a monosemantic basis —
  found by an SAE and read off Neuronpedia.

They're complementary, and the frontier is making them meet. Open directions from
here:

- Find an SAE feature that corresponds to a **circuit role** — e.g. a feature that
  fires exactly when a token is the indirect object, or the duplicated subject.
- Run the **patching from 01 in the SAE basis**: patch individual *features*
  clean→corrupted instead of raw activations, and localize the behavior to
  features rather than positions.
- Move up the stack: attention-SAEs, transcoders, or a larger model where
  superposition bites harder.

**Pointers.** [ARENA's SAE chapter](https://arena3-chapter1-transformer-interp.streamlit.app/),
[Neuronpedia](https://neuronpedia.org), and Anthropic's *Towards Monosemanticity*
and *Scaling Monosemanticity* for where this goes at scale.